# Activity Distance Exploration

Exploratory notebook for Figure 3 distance measurements. The current `FiringDistanceAnalyzer` compares distance-dependent spike activity to the ephaptic-axonal model. This notebook keeps the analysis separate from the figure assembly notebook: it first asks which distance observables create the clearest disagreement between an axonal-only monotone decay and the axonal-ephaptic model.

Working hypothesis:
- an axonal-only model should mostly produce smooth monotone decay with distance;
- ephaptic coupling adds frequency-dependent constructive/destructive distance bands, visible as residual structure after subtracting a monotone baseline;
- the strongest Figure 3 evidence should therefore come from residuals and null-normalized pair-distance enrichment, not from raw distance decay alone.

In [1]:
%matplotlib inline

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import pearsonr

repo_root = Path.cwd()
if not (repo_root / "ephax").exists() and (repo_root.parent / "ephax").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.environ.setdefault("MPLCONFIGDIR", str((repo_root / ".mpl-cache").resolve()))

from ephax import FiringDistanceAnalyzer, PrepConfig, RestingActivityDataset
from ephax.modeling.ephaptic import correlation_function

plt.rcParams["figure.dpi"] = 120
np.random.seed(0)

/Users/danielrebbin/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/danielrebbin/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Configuration

The variable names mirror `burst_analysis.ipynb` where possible. Start with one well so we can inspect the measurement behavior before deciding which panels belong in the reusable codebase.

In [2]:
# Dataset switch: "stim_removal_null" or "raw_h5".
DATASET = "stim_removal_null"

# Single-recording selection. DIV is used by stimRemovalNull and ignored for raw H5.
WELL = 0
DIV = 12
RAW_H5_DIV = 40

STIM_DATA_ROOT = repo_root / "ephax/data/stimRemovalNull"
STIM_DATA_FILENAME_TEMPLATE = "DIV{div}_240703_data_well{well}_exp_data.npz"

H5_FILE_INFO = {
    0: ("ephax/data", "well_0.raw.h5", 0),
    1: ("ephax/data", "well_1.raw.h5", 1),
    2: ("ephax/data", "well_2_3.raw.h5", 2),
    3: ("ephax/data", "well_2_3.raw.h5", 3),
    4: ("ephax/data", "well_4.raw.h5", 4),
    5: ("ephax/data", "well_5.raw.h5", 5),
}

AGGREGATE_WELLS = [0, 1, 2, 3, 4, 5]
AGGREGATE_DIVS = [DIV]

START_SEC = 0.0
END_SEC = 300.0 if DATASET == "stim_removal_null" else 600.0
MIN_AMP = 0.0

TOP_START = 0
TOP_STOP = 1000 if DATASET == "stim_removal_null" else 300

# Distance-analysis defaults.
MIN_DISTANCE_UM = 50.0
MAX_DISTANCE_UM = 3500.0
DISTANCE_BINS = 74
COFIRING_PM_MS = 2.0

# Existing model defaults in FiringDistanceAnalyzer.
V_EPH_M_PER_S = 0.1
V_AX_M_PER_S = 0.45
STD = 0.15
LAMBDA_EPH_UM = 100000.0

# Fixed frequency values can be copied from the aggregate IFR KDE section of burst_analysis.ipynb.
# Leave as None to let FiringDistanceAnalyzer use its legacy IFR peak path.
ACTIVITY_IFR_FREQUENCY_VALUES_HZ = None
# ACTIVITY_IFR_FREQUENCY_VALUES_HZ = np.array([45.324])

PERMUTE_FOR_NULL = True
PERM_SEED = 12345
SHUFFLE_LAYOUT_COORDINATES = True

In [3]:
def stim_recording_path(well, div):
    return STIM_DATA_ROOT / f"well{int(well)}" / STIM_DATA_FILENAME_TEMPLATE.format(div=int(div), well=int(well))


def build_recording_spec(well, div=None):
    if DATASET == "stim_removal_null":
        div = DIV if div is None else div
        path = stim_recording_path(well, div)
        if not path.exists():
            raise FileNotFoundError(f"Could not find stimRemovalNull recording file: {path}")
        return {
            "dataset": DATASET,
            "source": "npz",
            "path": path,
            "well": int(well),
            "div": int(div),
            "recording_id": f"stimRemovalNull_well{int(well)}_DIV{int(div)}",
            "label": f"stimRemovalNull well{int(well)} DIV{int(div)}",
            "source_file": path.name,
        }

    if DATASET == "raw_h5":
        if int(well) not in H5_FILE_INFO:
            raise ValueError(f"Unsupported raw H5 WELL={well}; available wells: {sorted(H5_FILE_INFO)}")
        h5_folder, h5_filename, h5_well_index = H5_FILE_INFO[int(well)]
        folder_path = repo_root / h5_folder
        file_path = folder_path / h5_filename
        if not file_path.exists():
            raise FileNotFoundError(f"Could not find raw H5 recording file: {file_path}")
        return {
            "dataset": DATASET,
            "source": "h5",
            "path": file_path,
            "folder": folder_path,
            "filename": h5_filename,
            "well": int(h5_well_index),
            "requested_well": int(well),
            "div": int(RAW_H5_DIV),
            "recording_id": f"raw_h5_well{int(well)}_DIV{int(RAW_H5_DIV)}",
            "label": f"raw H5 well{int(well)} DIV{int(RAW_H5_DIV)}",
            "source_file": h5_filename,
        }

    raise ValueError("DATASET must be 'stim_removal_null' or 'raw_h5'.")


def load_recording_from_spec(spec, start_sec=START_SEC, end_sec=END_SEC, min_amp=MIN_AMP):
    if spec["source"] == "npz":
        file_info = [(str(spec["path"]), float(start_sec), float(end_sec), int(spec["well"]))]
        return RestingActivityDataset.from_file_info(file_info, source="npz", min_amp=float(min_amp))

    if spec["source"] == "h5":
        file_info = [
            (str(spec["folder"]), str(spec["filename"]), float(start_sec), float(end_sec), int(spec["well"])),
        ]
        return RestingActivityDataset.from_file_info(file_info, source="h5", min_amp=float(min_amp))

    raise ValueError(f"Unsupported source: {spec['source']}")


def make_permuted_dataset(ds, seed=PERM_SEED, shuffle_layout_coordinates=SHUFFLE_LAYOUT_COORDINATES):
    rng = np.random.default_rng(seed)
    permuted = []
    for rec_idx, rec in enumerate(ds.recordings):
        permuted.append(
            rec.randomize_electrode_mapping(
                rng=np.random.default_rng(int(rng.integers(0, 2**31 - 1)) + rec_idx),
                inplace=False,
                shuffle_layout_coordinates=shuffle_layout_coordinates,
            )
        )
    return RestingActivityDataset(recordings=permuted, sf=ds.sf)

## Load Data and Select References

In [4]:
recording_spec = build_recording_spec(WELL, DIV)
ds = load_recording_from_spec(recording_spec)
rec = ds.recordings[0]

duration = float(rec.end_time - rec.start_time)
in_window = (rec.spikes["time"] >= rec.start_time) & (rec.spikes["time"] <= rec.end_time)
print(f"Loaded: {recording_spec['label']}")
print(f"Source: {recording_spec['source_file']}")
print(f"Window: [{rec.start_time:.3f}, {rec.end_time:.3f}] s ({duration:.1f} s)")
print(f"Spikes in window: {int(np.sum(in_window)):,}")
print(f"Active electrodes in window: {np.unique(rec.spikes['electrode'][in_window]).size:,}")

prep_cfg = PrepConfig(mode="top", top_start=TOP_START, top_stop=TOP_STOP, verbose=False)
refs = ds.select_ref_electrodes(prep_cfg)[0]
print(f"Selected reference electrodes: {refs.size}")

perm_ds = make_permuted_dataset(ds) if PERMUTE_FOR_NULL else None
frequency_values = None if ACTIVITY_IFR_FREQUENCY_VALUES_HZ is None else np.asarray(ACTIVITY_IFR_FREQUENCY_VALUES_HZ, dtype=float)

fd = FiringDistanceAnalyzer(
    ds,
    dataset_perm=perm_ds,
    refs_per_recording=[refs],
    selection_prep_config=prep_cfg,
    v_eph=V_EPH_M_PER_S,
    v_ax=V_AX_M_PER_S,
    std=STD,
    lambda_eph=LAMBDA_EPH_UM,
    frequency_values_hz=frequency_values,
)

Loaded: stimRemovalNull well0 DIV12
Source: DIV12_240703_data_well0_exp_data.npz
Window: [0.000, 300.000] s (300.0 s)
Spikes in window: 129,192
Active electrodes in window: 998
Selected reference electrodes: 998


## Existing Figure-3 Measurements

These are the three existing distance summaries that `FiringDistanceAnalyzer` supports:

- `avg_rate_vs_distance`: rate of active target electrodes as a function of distance from selected reference electrodes.
- `cofiring_avg_vs_distance`: expected target spikes within `plusminus_ms` around each reference spike, binned by reference-target distance.
- `distance_histogram`: pairwise distances among selected active/reference electrodes, optionally finite-size corrected and normalized by a permuted layout null.

The key question for Figure 3 is not whether activity decreases with distance; axonal-only models can explain that. The stronger test is whether the residuals after a monotone distance baseline show distance bands predicted by the ephaptic-axonal timing model.

In [5]:
rate_res = fd.avg_rate_vs_distance(log=False, min_distance=MIN_DISTANCE_UM, max_distance=MAX_DISTANCE_UM)
cof_res = fd.cofiring_avg_vs_distance(
    plusminus_ms=COFIRING_PM_MS,
    log=False,
    min_distance=MIN_DISTANCE_UM,
    max_distance=MAX_DISTANCE_UM,
)
dist_vals, dist_weights = fd.distance_histogram(
    finite_size_correction=True,
    min_distance=MIN_DISTANCE_UM,
    max_distance=MAX_DISTANCE_UM,
    bins=DISTANCE_BINS,
)

print(f"Rate pairs: {rate_res.distances.size:,} raw pairs | binned points: {rate_res.binned.centers.size}")
print(f"Co-firing pairs: {cof_res.distances.size:,} raw pairs | binned points: {cof_res.binned.centers.size}")
print(f"Selected-electrode pair distances: {dist_vals.size:,}")

KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), constrained_layout=True)

axes[0].plot(rate_res.binned.centers, rate_res.binned.mean, color="tab:blue")
axes[0].fill_between(
    rate_res.binned.centers,
    rate_res.binned.mean - rate_res.binned.stderr,
    rate_res.binned.mean + rate_res.binned.stderr,
    color="tab:blue",
    alpha=0.25,
)
axes[0].set_title("Firing rate vs distance")
axes[0].set_xlabel("distance (um)")
axes[0].set_ylabel("rate (Hz)")

axes[1].plot(cof_res.binned.centers, cof_res.binned.mean, color="tab:green")
axes[1].fill_between(
    cof_res.binned.centers,
    cof_res.binned.mean - cof_res.binned.stderr,
    cof_res.binned.mean + cof_res.binned.stderr,
    color="tab:green",
    alpha=0.25,
)
axes[1].set_title(f"Co-firing vs distance (+/- {COFIRING_PM_MS:g} ms)")
axes[1].set_xlabel("distance (um)")
axes[1].set_ylabel("spikes per reference spike")

hist_edges = np.linspace(MIN_DISTANCE_UM, MAX_DISTANCE_UM, DISTANCE_BINS + 1)
axes[2].hist(dist_vals, bins=hist_edges, weights=dist_weights, color="0.35", alpha=0.8)
axes[2].set_title("Selected-electrode pair distances")
axes[2].set_xlabel("distance (um)")
axes[2].set_ylabel("finite-size corrected count")

plt.show()

## Axonal-Only Baseline: Smooth Monotone Distance Decay

For this exploratory notebook, the axonal-only baseline is deliberately conservative: fit a smooth exponential-plus-offset to the binned data, then analyze residuals. This does not claim that the fitted exponential is the final biological axonal model. It is a practical null for the visual question: what remains after the obvious monotone distance dependence is removed?

In [ ]:
def exp_decay(x, amplitude, decay_per_um, offset):
    return amplitude * np.exp(-decay_per_um * x) + offset


def fit_exp_decay(x, y, yerr=None):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    if yerr is not None:
        yerr = np.asarray(yerr, dtype=float)
        valid &= np.isfinite(yerr) & (yerr > 0)
    x_fit = x[valid]
    y_fit = y[valid]
    if x_fit.size < 4:
        return None
    amp0 = max(1e-12, float(np.nanmax(y_fit) - np.nanmin(y_fit)))
    decay0 = 1.0 / max(1.0, float(np.nanmax(x_fit) - np.nanmin(x_fit)))
    offset0 = max(0.0, float(np.nanmin(y_fit)))
    sigma = None if yerr is None else yerr[valid]
    bounds = ([0.0, 0.0, -np.inf], [np.inf, np.inf, np.inf])
    popt, pcov = curve_fit(
        exp_decay,
        x_fit,
        y_fit,
        p0=[amp0, decay0, offset0],
        sigma=sigma,
        absolute_sigma=False,
        bounds=bounds,
        maxfev=10000,
    )
    fitted = exp_decay(x, *popt)
    residual = y - fitted
    return {"params": popt, "cov": pcov, "fitted": fitted, "residual": residual, "valid": valid}


rate_fit = fit_exp_decay(rate_res.binned.centers, rate_res.binned.mean, rate_res.binned.stderr)
cof_fit = fit_exp_decay(cof_res.binned.centers, cof_res.binned.mean, cof_res.binned.stderr)

for name, fit in [("rate", rate_fit), ("cofiring", cof_fit)]:
    if fit is None:
        print(f"{name}: exponential fit unavailable")
    else:
        amp, decay, offset = fit["params"]
        print(f"{name}: amplitude={amp:.4g}, decay={decay:.4g} 1/um, offset={offset:.4g}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7.5), sharex="col", constrained_layout=True)

for col, (label, res, fit, color, ylabel) in enumerate([
    ("Firing rate", rate_res, rate_fit, "tab:blue", "rate (Hz)"),
    ("Co-firing", cof_res, cof_fit, "tab:green", "spikes/ref spike"),
]):
    ax = axes[0, col]
    ax.plot(res.binned.centers, res.binned.mean, color=color, marker="o", ms=3, label="data")
    ax.fill_between(res.binned.centers, res.binned.mean - res.binned.stderr, res.binned.mean + res.binned.stderr, color=color, alpha=0.2)
    if fit is not None:
        ax.plot(res.binned.centers, fit["fitted"], color="black", lw=1.5, label="axonal-only baseline")
    ax.set_title(label)
    ax.set_ylabel(ylabel)
    ax.legend(loc="best")

    axr = axes[1, col]
    if fit is not None:
        stderr = np.asarray(res.binned.stderr, dtype=float)
        resid_z = np.divide(fit["residual"], stderr, out=np.full_like(fit["residual"], np.nan), where=stderr > 0)
        axr.axhline(0.0, color="0.5", lw=1.0)
        axr.plot(res.binned.centers, fit["residual"], color="purple", marker="o", ms=3, label="raw residual")
        axr2 = axr.twinx()
        axr2.plot(res.binned.centers, resid_z, color="0.25", lw=1.0, alpha=0.6, label="residual / stderr")
        axr2.set_ylabel("residual z-like")
    axr.set_xlabel("distance (um)")
    axr.set_ylabel("residual")

plt.show()

## Ephaptic-Axonal Distance Prediction

`correlation_function(r, hz, v_eph, v_ax, lambda_eph)` predicts a signed, frequency-dependent distance envelope. The constructive/destructive bands come from the relative arrival-time term:

```text
delta_t(r) = r * (1 / v_eph - 1 / v_ax)
cos(2*pi*f*delta_t)
```

With `v_eph=0.1 m/s` and `v_ax=0.45 m/s`, the first constructive/disagreement windows can sit in a few-hundred-micrometer range for ~40-50 Hz activity. This is where raw monotone distance decay is least diagnostic and residual structure is most informative.

In [ ]:
def resolve_model_frequencies(analyzer, peak_min_hz=30.0, peak_max_hz=1000.0):
    ok, gamma_hz, weights = analyzer._compute_ifr_peaks_weights(peak_min_hz=peak_min_hz, peak_max_hz=peak_max_hz)
    if not ok:
        return np.array([], dtype=float), np.array([], dtype=float)
    weights = np.asarray(weights, dtype=float)
    weights = weights / max(1e-12, np.nanmax(np.abs(weights)))
    return np.asarray(gamma_hz, dtype=float), weights


def ephaptic_curve_at(dist_um, freqs_hz, weights, v_eph_m_s=V_EPH_M_PER_S, v_ax_m_s=V_AX_M_PER_S, lambda_eph_um=LAMBDA_EPH_UM):
    dist_um = np.asarray(dist_um, dtype=float)
    out = np.zeros_like(dist_um, dtype=float)
    v_eph_um_s = float(v_eph_m_s) * 1e6
    v_ax_um_s = float(v_ax_m_s) * 1e6
    for hz, weight in zip(freqs_hz, weights):
        out += float(weight) * correlation_function(dist_um, float(hz), v_eph_um_s, v_ax_um_s, float(lambda_eph_um))
    return out


model_freqs_hz, model_weights = resolve_model_frequencies(fd)
print("Model frequencies (Hz):", np.round(model_freqs_hz, 3))
print("Relative weights:", np.round(model_weights, 3))

r_grid = np.linspace(MIN_DISTANCE_UM, MAX_DISTANCE_UM, 1500)
model_curve = ephaptic_curve_at(r_grid, model_freqs_hz, model_weights)

fig, ax = plt.subplots(figsize=(10, 3.8), constrained_layout=True)
ax.plot(r_grid, model_curve, color="crimson", lw=1.5)
ax.axhline(0.0, color="0.5", lw=1.0)
ax.set_xlabel("distance (um)")
ax.set_ylabel("signed ephaptic-axonal predictor")
ax.set_title("Distance bands where ephaptic-axonal model departs from monotone axonal baseline")
plt.show()

## Where Should the Models Disagree Most?

For binned firing/co-firing data, the strongest disagreement candidates are bins with:

- large absolute ephaptic predictor after removing the monotone baseline;
- residuals whose sign agrees with the ephaptic predictor;
- enough samples / small enough uncertainty;
- ideally reproducibility across rate, co-firing, and pair-distance ratio.

The table below is exploratory. It ranks distance bins by `abs(model) * abs(residual / stderr)`, and also reports sign agreement.

In [ ]:
def disagreement_table(label, result, fit, freqs_hz, weights, top_n=12):
    if fit is None or freqs_hz.size == 0:
        return pd.DataFrame()
    x = np.asarray(result.binned.centers, dtype=float)
    y = np.asarray(result.binned.mean, dtype=float)
    err = np.asarray(result.binned.stderr, dtype=float)
    residual = np.asarray(fit["residual"], dtype=float)
    predictor = ephaptic_curve_at(x, freqs_hz, weights)
    pred_scale = np.nanmax(np.abs(predictor))
    predictor_norm = predictor / pred_scale if pred_scale > 0 else predictor
    resid_z = np.divide(residual, err, out=np.full_like(residual, np.nan), where=err > 0)
    score = np.abs(predictor_norm) * np.abs(resid_z)
    sign_agreement = np.sign(predictor_norm) == np.sign(residual)
    out = pd.DataFrame(
        {
            "metric": label,
            "distance_um": x,
            "observed": y,
            "axonal_baseline": fit["fitted"],
            "residual": residual,
            "residual_z": resid_z,
            "ephaptic_predictor_norm": predictor_norm,
            "sign_agreement": sign_agreement,
            "disagreement_score": score,
        }
    )
    return out.sort_values("disagreement_score", ascending=False).head(top_n).reset_index(drop=True)


disagreement = pd.concat(
    [
        disagreement_table("rate", rate_res, rate_fit, model_freqs_hz, model_weights),
        disagreement_table("cofiring", cof_res, cof_fit, model_freqs_hz, model_weights),
    ],
    ignore_index=True,
)
disagreement

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, constrained_layout=True)

for ax, label, res, fit, color in [
    (axes[0], "rate", rate_res, rate_fit, "tab:blue"),
    (axes[1], "cofiring", cof_res, cof_fit, "tab:green"),
]:
    if fit is None:
        continue
    x = np.asarray(res.binned.centers, dtype=float)
    residual = np.asarray(fit["residual"], dtype=float)
    predictor = ephaptic_curve_at(x, model_freqs_hz, model_weights)
    pred_scale = np.nanmax(np.abs(predictor))
    predictor_norm = predictor / pred_scale if pred_scale > 0 else predictor
    resid_scale = np.nanmax(np.abs(residual))
    residual_norm = residual / resid_scale if resid_scale > 0 else residual

    ax.axhline(0.0, color="0.5", lw=1.0)
    ax.plot(x, residual_norm, color=color, marker="o", ms=3, label=f"{label} residual, normalized")
    ax.plot(x, predictor_norm, color="crimson", lw=1.5, label="ephaptic predictor, normalized")
    ax.set_ylabel("normalized value")
    ax.set_title(f"{label}: residual structure vs ephaptic-axonal distance prediction")
    finite = np.isfinite(residual_norm) & np.isfinite(predictor_norm)
    if np.count_nonzero(finite) >= 3:
        r, p = pearsonr(predictor_norm[finite], residual_norm[finite])
        ax.text(0.01, 0.93, f"r={r:.3f}, p={p:.2e}", transform=ax.transAxes, ha="left", va="top")
    ax.legend(loc="best")

axes[-1].set_xlabel("distance (um)")
plt.show()

## Pair-Distance Ratio Against a Spatial Null

If ephaptic coupling changes which electrodes become strongly active together, the selected-electrode pair distance distribution should be enriched at constructive distances relative to a shuffled-layout null. This observable can be more Figure-3-friendly than raw firing rate because it is explicitly spatial and null-normalized.

In [ ]:
def pair_distance_ratio(analyzer, bins=DISTANCE_BINS):
    edges = np.linspace(MIN_DISTANCE_UM, MAX_DISTANCE_UM, int(bins) + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    distances, weights = analyzer.distance_histogram(
        finite_size_correction=True,
        min_distance=MIN_DISTANCE_UM,
        max_distance=MAX_DISTANCE_UM,
        bins=bins,
        dataset=analyzer.ds,
    )
    counts, _ = np.histogram(distances, bins=edges, weights=weights)
    if analyzer.ds_perm is None:
        return centers, counts, np.full_like(counts, np.nan, dtype=float), np.full_like(counts, np.nan, dtype=float)
    perm_distances, perm_weights = analyzer.distance_histogram(
        finite_size_correction=True,
        min_distance=MIN_DISTANCE_UM,
        max_distance=MAX_DISTANCE_UM,
        bins=bins,
        dataset=analyzer.ds_perm,
    )
    perm_counts, _ = np.histogram(perm_distances, bins=edges, weights=perm_weights)
    ratio = np.divide(counts, perm_counts, out=np.full_like(counts, np.nan, dtype=float), where=perm_counts > 0)
    return centers, counts, perm_counts, ratio


pair_centers, pair_counts, pair_perm_counts, pair_ratio = pair_distance_ratio(fd)
pair_predictor = ephaptic_curve_at(pair_centers, model_freqs_hz, model_weights)
pair_predictor_norm = pair_predictor / np.nanmax(np.abs(pair_predictor)) if np.nanmax(np.abs(pair_predictor)) > 0 else pair_predictor

fig, axes = plt.subplots(1, 2, figsize=(14, 4.3), constrained_layout=True)
axes[0].plot(pair_centers, pair_counts, color="black", label="observed")
if np.isfinite(pair_perm_counts).any():
    axes[0].plot(pair_centers, pair_perm_counts, color="0.6", label="permuted null")
axes[0].set_title("Selected-electrode pair distances")
axes[0].set_xlabel("distance (um)")
axes[0].set_ylabel("finite-size corrected count")
axes[0].legend(loc="best")

axes[1].axhline(1.0, color="0.5", lw=1.0)
axes[1].plot(pair_centers, pair_ratio, color="purple", marker="o", ms=3, label="observed / permuted")
ax2 = axes[1].twinx()
ax2.plot(pair_centers, pair_predictor_norm, color="crimson", lw=1.5, alpha=0.8, label="ephaptic predictor")
axes[1].set_title("Pair-distance enrichment vs ephaptic predictor")
axes[1].set_xlabel("distance (um)")
axes[1].set_ylabel("observed / permuted")
ax2.set_ylabel("normalized predictor")

finite = np.isfinite(pair_ratio) & np.isfinite(pair_predictor_norm)
if np.count_nonzero(finite) >= 3:
    r, p = pearsonr(pair_predictor_norm[finite], pair_ratio[finite])
    axes[1].text(0.01, 0.93, f"r={r:.3f}, p={p:.2e}", transform=axes[1].transAxes, ha="left", va="top")

lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, loc="best")
plt.show()

## Practical Readout Choice

The most suitable Figure 3 measurements should be those where an axonal-only explanation has the least room to hide:

1. **Co-firing residual vs distance**: fit a monotone baseline, then test whether residual peaks/troughs align with ephaptic-axonal constructive/destructive bands.
2. **Selected-electrode pair-distance ratio**: observed active/reference-electrode pair distances divided by a spatially permuted layout null; this directly asks whether strongly active electrodes are over-represented at predicted distances.
3. **Firing-rate residual vs distance**: useful secondary support, but weaker because firing rate can have many non-spatial confounds.

For integration into the codebase, the next step should be to turn the helper logic above into pure metrics that return compact tables: residuals, normalized model predictors, sign agreement, and null-normalized pair-distance ratios. The final figure notebook can then pull those tables and draw polished multipanel visuals.